# 00_env_config — shared FabricOps runtime configuration

This notebook defines reusable environment paths, runtime naming policy, AI prompt settings, quality defaults, governance defaults, and lineage defaults. Other exploration and pipeline notebooks reuse this configuration instead of redefining paths manually.

**You edit**
- Environment names
- Lakehouse / warehouse targets
- Allowed notebook prefixes
- AI prompt templates
- Quality and governance defaults

**This notebook produces**
- Validated framework config
- Resolved Fabric paths
- Startup smoke-test evidence
- AI availability check


## Segment 1: Import configuration helpers

What this does: imports the public FabricOps functions used to define and validate shared runtime config.

Functions used:
- `create_*_config` — build typed config blocks
- `validate_framework_config` — validate assembled config
- `run_config_smoke_tests` — startup readiness checks

User edits:
- None in this segment

Output:
- Imported helpers available for the notebook


In [ ]:
from fabricops_kit.environment_config import (
    bootstrap_fabric_env,
    check_fabric_ai_functions_available,
    configure_fabric_ai_functions,
    create_ai_prompt_config,
    create_framework_config,
    create_governance_config,
    create_lineage_config,
    create_notebook_runtime_config,
    create_path_config,
    create_quality_config,
    get_path,
    load_fabric_config,
    run_config_smoke_tests,
    validate_framework_config,
)
from fabricops_kit.fabric_input_output import Housepath


## User-editable project constants
Replace all placeholders before use.


## Segment 2: Define environment targets and notebook policy

What this does: sets environment name, lakehouse/warehouse paths, and allowed notebook prefixes.

Functions used:
- `create_path_config` — declares target storage per environment
- `create_notebook_runtime_config` — notebook naming policy
- `Housepath` — lakehouse address details

User edits:
- `ENV_NAME`, `PATH_CONFIG`, and prefix list

Output:
- Reusable environment and runtime settings


In [ ]:
ENV_NAME = "dev"
PATH_CONFIG = create_path_config({ENV_NAME: {
    "source": Housepath("00000000-0000-0000-0000-000000000001", "11111111-1111-1111-1111-111111111111", "lh_source", "abfss://REPLACE-WORKSPACE@onelake.dfs.fabric.microsoft.com/REPLACE-HOUSE"),
    "unified": Housepath("00000000-0000-0000-0000-000000000001", "11111111-1111-1111-1111-111111111112", "lh_unified", "abfss://REPLACE-WORKSPACE@onelake.dfs.fabric.microsoft.com/REPLACE-HOUSE"),
    "product": Housepath("00000000-0000-0000-0000-000000000001", "11111111-1111-1111-1111-111111111113", "lh_product", "abfss://REPLACE-WORKSPACE@onelake.dfs.fabric.microsoft.com/REPLACE-HOUSE"),
    "metadata": Housepath("00000000-0000-0000-0000-000000000001", "11111111-1111-1111-1111-111111111114", "lh_metadata", "abfss://REPLACE-WORKSPACE@onelake.dfs.fabric.microsoft.com/REPLACE-HOUSE"),
}})


In [ ]:
NOTEBOOK_RUNTIME_CONFIG = create_notebook_runtime_config(allowed_notebook_prefixes=["00_", "02_ex_", "03_pc_"])


## Segment 3: Set AI, quality, governance, and lineage defaults

What this does: defines prompt templates and deterministic default behavior used by exploration and pipeline notebooks.

Functions used:
- `create_ai_prompt_config`, `create_quality_config`, `create_governance_config`, `create_lineage_config`

User edits:
- Prompt text and default thresholds/policies

Output:
- Policy defaults bundled into config components


In [ ]:
AI_PROMPT_CONFIG = create_ai_prompt_config(
    dq_rule_candidate_template="Suggest deterministic DQ rule candidates as JSON. Profile: {profile}",
    governance_candidate_template="Suggest governance labels as JSON. Profile: {profile}",
    handover_summary_template="Summarize run handover details as markdown. Context: {context}",
)


In [ ]:
GOVERNANCE_CONFIG = create_governance_config(required_classification=True)
QUALITY_CONFIG = create_quality_config(default_severity="warning", fail_on_critical=True)
LINEAGE_CONFIG = create_lineage_config(capture_transformation_steps=True, capture_ai_summaries=False)


## Segment 4: Assemble and validate framework config

What this does: combines all config blocks into one framework object and validates it.

Functions used:
- `create_framework_config`
- `validate_framework_config`

User edits:
- Optional validation expectations

Output:
- `CONFIG` ready for downstream notebook reuse


In [ ]:
CONFIG = create_framework_config(path_config=PATH_CONFIG, notebook_runtime_config=NOTEBOOK_RUNTIME_CONFIG, ai_prompt_config=AI_PROMPT_CONFIG, governance_config=GOVERNANCE_CONFIG, quality_config=QUALITY_CONFIG, lineage_config=LINEAGE_CONFIG)


In [ ]:
CONFIG = validate_framework_config(CONFIG)
FABRIC_CONFIG = load_fabric_config(CONFIG)


## Segment 5: Run startup checks and show resolved paths

What this does: runs smoke checks and prints resolved Fabric paths used by templates.

Functions used:
- `run_config_smoke_tests`
- `get_path`

User edits:
- `required_targets` list for your environment

Output:
- Startup evidence and concrete resolved paths


In [ ]:
print(run_config_smoke_tests(config=FABRIC_CONFIG, env=ENV_NAME, required_targets=["source", "unified", "product", "metadata"]))
print(bootstrap_fabric_env(config=FABRIC_CONFIG, env=ENV_NAME, required_targets=["source", "unified", "product", "metadata"]))
if check_fabric_ai_functions_available().get("available"):
    configure_fabric_ai_functions(temperature=0.0)


In [ ]:
print(get_path(ENV_NAME, "source", config=FABRIC_CONFIG))
print(get_path(ENV_NAME, "unified", config=FABRIC_CONFIG))
print(get_path(ENV_NAME, "product", config=FABRIC_CONFIG))
print(get_path(ENV_NAME, "metadata", config=FABRIC_CONFIG))
